## Online Shopping Demand Analysis by Zip Code

This notebook cleans and uses the `uszips.csv` dataset to generate a rough heuristic for online shopping demand for each continental U.S. zip code. Each zip code's demand score is combined with its geographic centroid to create the `demandRegion` dataset.

The `demandRegion` dataset maps centroid coordinates to demand scores.

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

## EDA and Cleaning of uszips.csv

In [2]:
uszips_filepath = "../Data/DemandRegion/uszips.csv"

df = pd.read_csv(uszips_filepath)
df.head()

,zip,lat,lng,city,state_id,state_name,zcta,parent_zcta,population,density,county_fips,county_name,county_weights,county_names_all,county_fips_all,imprecise,military,timezone
0,601,18.18027,-66.75266,Adjuntas,PR,Puerto Rico,True,NaN,16721.0,100.2,72001,Adjuntas,"{""72001"": 98.74, ""72141"": 1.26}",Adjuntas|Utuado,72001|72141,False,False,America/Puerto_Rico
1,602,18.36075,-67.17541,Aguada,PR,Puerto Rico,True,NaN,37510.0,477.6,72003,Aguada,"{""72003"": 100}",Aguada,72003,False,False,America/Puerto_Rico
2,603,18.45744,-67.12225,Aguadilla,PR,Puerto Rico,True,NaN,48317.0,543.1,72005,Aguadilla,"{""72005"": 99.76, ""72099"": 0.24}",Aguadilla|Moca,72005|72099,False,False,America/Puerto_Rico
3,606,18.16585,-66.93716,Maricao,PR,Puerto Rico,True,NaN,5435.0,47.3,72093,Maricao,"{""72093"": 82.26, ""72153"": 11.67, ""72121"": 6.06}",Maricao|Yauco|Sabana Grande,72093|72153|72121,False,False,America/Puerto_Rico
4,610,18.29110,-67.12243,Anasco,PR,Puerto Rico,True,NaN,25413.0,264.4,72011,Añasco,"{""72011"": 96.8, ""72099"": 2.83, ""72083"": 0.37}",Añasco|Moca|Las Marías,72011|72099|72083,False,False,America/Puerto_Rico


In [3]:
df.shape

(33782, 18)

33,782 Entries

In [4]:
len(df["zip"].unique())

33782

No duplicated zip-codes. Good!

In [5]:
states = df["state_id"].unique()
print("Number of 'States':", len(states))

print(states)

Number of 'States': 56
['PR' 'VI' 'MA' 'RI' 'NH' 'ME' 'VT' 'CT' 'NY' 'NJ' 'PA' 'DE' 'DC' 'VA'
 'MD' 'WV' 'NC' 'SC' 'GA' 'FL' 'AL' 'TN' 'MS' 'KY' 'OH' 'IN' 'MI' 'IA'
 'WI' 'MN' 'SD' 'ND' 'MT' 'IL' 'MO' 'KS' 'NE' 'LA' 'AR' 'OK' 'TX' 'CO'
 'WY' 'ID' 'UT' 'AZ' 'NM' 'NV' 'CA' 'HI' 'AS' 'GU' 'MP' 'OR' 'WA' 'AK']


The following codes correspond to non-continental US:
* AK (Alaska)
* HI (Hawaii)
* PR (Puerto Rico)
* VI (U.S. Virgin Islands)
* AS (American Samoa)
* GU (Guam)
* MP (Northern Mariana Islands)

Let's remove them for now and just focus on continental US.

In [6]:
non_continental_codes = ['AK', 'HI', 'PR', 'VI', 'AS', 'GU', 'MP']
continental_df = df[~df['state_id'].isin(non_continental_codes)]

In [7]:
continental_df[continental_df['population'].isna()]

,zip,lat,lng,city,state_id,state_name,zcta,parent_zcta,population,density,county_fips,county_name,county_weights,county_names_all,county_fips_all,imprecise,military,timezone
2998,11717,40.78405,-73.25223,Brentwood,NY,New York,True,NaN,NaN,NaN,36103,Suffolk,"{""36103"": 100}",Suffolk,36103,False,False,America/New_York
3062,11798,40.75200,-73.37497,Wyandanch,NY,New York,True,NaN,NaN,NaN,36103,Suffolk,"{""36103"": 100}",Suffolk,36103,False,False,America/New_York


Two zip codes don't have population values. We will use population to estimate demand score. Given it's only two, we impute manually with the most recent available census data. Found on https://www.unitedstateszipcodes.org/

In [8]:
continental_df.loc[continental_df['zip'] == 11717, 'population'] = 64969 # From 2022 Census
continental_df.loc[continental_df['zip'] == 11798, 'population'] = 17215 # From 2020 Census

continental_df.loc[continental_df['zip'] == 11717, 'density'] = 5788 # From 2022 Census
continental_df.loc[continental_df['zip'] == 11798, 'density'] = 3458 # From 2020 Census

In [9]:
columns_to_keep = ['zip', 'lat', 'lng', 'population', 'density', ]
zip_codes_df = continental_df[columns_to_keep]

zip_codes_df.head()

,zip,lat,lng,population,density
137,1001,42.06262,-72.62521,16136.0,551.7
138,1002,42.37633,-72.46462,24726.0,179.3
139,1003,42.39135,-72.52327,12458.0,5981.7
140,1005,42.42117,-72.10655,4786.0,42.8
141,1007,42.28163,-72.40009,15406.0,108.4


## Retrieving Relevant Socioeconomic Data by Zip-Code for Estimating Online Shopping Demand

In [10]:
import requests

# ACS 5-year detailed table endpoint
base_url = "https://api.census.gov/data/2021/acs/acs5"

# Request median income, median age, and median household size
params = {
    "get": "NAME,B19013_001E,B01002_001E,B25010_001E",
    "for": "zip code tabulation area:*"
}

resp = requests.get(base_url, params=params)
data = resp.json()

In [11]:
df_socioeconomic = pd.DataFrame(data[1:], columns=data[0])
df_socioeconomic.rename(
        columns={
            'B19013_001E': 'median_household_income',
            'B01002_001E': 'median_age',
            'B25010_001E': 'median_household_size',
            'zip code tabulation area': 'zip'
            }, 
        inplace=True
    )

df_socioeconomic["median_household_income"] = df_socioeconomic["median_household_income"].astype(float)
df_socioeconomic["median_age"] = df_socioeconomic["median_age"].astype(float)
df_socioeconomic["median_household_size"] = df_socioeconomic["median_household_size"].astype(float)
df_socioeconomic["zip"] = df_socioeconomic["zip"].astype(int)
df_socioeconomic.drop(columns="NAME", inplace=True)

df_socioeconomic.head()

,median_household_income,median_age,median_household_size,zip
0,15292.0,43.7,3.16,601
1,18716.0,44.4,2.94,602
2,16789.0,44.1,2.49,603
3,18835.0,44.9,2.91,606
4,21239.0,43.5,2.92,610


### Clean this data

In [12]:
df_socioeconomic.describe()

,median_household_income,median_age,median_household_size,zip
count,3.377400e+04,3.377400e+04,3.377400e+04,33774.000000
mean,-6.138658e+07,-1.752823e+07,-2.268017e+07,49697.743856
std,1.928676e+08,1.066706e+08,1.208559e+08,27546.331750
min,-6.666667e+08,-6.666667e+08,-6.666667e+08,601.000000
25%,4.445950e+04,3.660000e+01,2.260000e+00,26733.000000
50%,5.875000e+04,4.160000e+01,2.500000e+00,49720.500000
75%,7.625000e+04,4.780000e+01,2.770000e+00,72159.250000
max,2.500010e+05,9.400000e+01,2.500000e+01,99929.000000


In [13]:
print(df_socioeconomic["median_household_income"][df_socioeconomic["median_household_income"] < 0].unique())
print(df_socioeconomic["median_age"][df_socioeconomic["median_age"] < 0].unique())
print(df_socioeconomic["median_household_size"][df_socioeconomic["median_household_size"] < 0].unique())

[-6.66666666e+08]
[-6.66666666e+08]
[-6.66666666e+08]


-6.66666666e+08 This value represents missing values, let's just make those NaN for now

In [14]:
df_socioeconomic.loc[df_socioeconomic["median_household_income"] == -666666666.0, "median_household_income"] = pd.NA
df_socioeconomic.loc[df_socioeconomic["median_age"] == -666666666.0, "median_age"] = pd.NA
df_socioeconomic.loc[df_socioeconomic["median_household_size"] == -666666666.0, "median_household_size"] = pd.NA


In [15]:
df_socioeconomic.isna().sum()

median_household_income    3113
median_age                  888
median_household_size      1149
zip                           0
dtype: int64

In [16]:
df_socioeconomic.head()

,median_household_income,median_age,median_household_size,zip
0,15292.0,43.7,3.16,601
1,18716.0,44.4,2.94,602
2,16789.0,44.1,2.49,603
3,18835.0,44.9,2.91,606
4,21239.0,43.5,2.92,610


## Join Socioeconomic Factors with Original Zip Codes Data

In [17]:
joined_df = pd.merge(zip_codes_df, df_socioeconomic, how="left", on="zip")
joined_df.head()

,zip,lat,lng,population,density,median_household_income,median_age,median_household_size
0,1001,42.06262,-72.62521,16136.0,551.7,72444.0,45.4,2.30
1,1002,42.37633,-72.46462,24726.0,179.3,65013.0,25.3,2.39
2,1003,42.39135,-72.52327,12458.0,5981.7,NaN,19.8,2.13
3,1005,42.42117,-72.10655,4786.0,42.8,103477.0,43.2,2.69
4,1007,42.28163,-72.40009,15406.0,108.4,101076.0,42.8,2.59


In [18]:
joined_df.isna().sum()

zip                           0
lat                           0
lng                           0
population                    0
density                       0
median_household_income    3058
median_age                  871
median_household_size      1120
dtype: int64

In [19]:
for col in ["median_household_income", "median_age", "median_household_size"]:
    joined_df[col] = joined_df[col].fillna(joined_df[col].median())

In [20]:
joined_df.describe()

,zip,lat,lng,population,density,median_household_income,median_age,median_household_size
count,33292.000000,33292.000000,33292.000000,33292.000000,33292.000000,33292.000000,33292.000000,33292.000000
mean,49385.799201,38.778739,-90.474304,9918.344197,510.096405,66937.534483,42.915709,2.564405
std,27113.252585,4.751856,13.688097,14937.613336,1952.412813,27741.267914,9.478517,0.556315
min,1001.000000,24.585500,-124.628160,0.000000,0.000000,2499.000000,0.000000,1.000000
25%,26803.500000,35.429992,-97.013503,653.750000,7.400000,50625.000000,37.200000,2.300000
50%,49453.500000,39.476165,-88.103990,2667.000000,30.600000,61527.000000,41.900000,2.520000
75%,71671.750000,42.057533,-80.259668,13330.000000,268.300000,76250.000000,47.900000,2.760000
max,99403.000000,49.099070,-67.018340,137213.000000,62798.400000,250001.000000,94.000000,25.000000


## Create Demand Score Heuristic

In [21]:
joined_df['demand_score'] = (
    ( # Estimate online shopping demand per household
        1.2 * (0.000026 * joined_df['median_household_income'] - 1.65) + 
        0.95 * (-0.083 * joined_df['median_age'] + 4.66) + 
        0.8 * (0.93 * joined_df['median_household_size'] - 1) +
        0.23 * np.where(
            joined_df["density"] < 1,
            -0.27 * joined_df["density"] + 0.17,
            (0.25/29) * joined_df["density"] - 0.1
        )
    ) 
    # Multiply by estimate of the number of households
    * (joined_df['population'] / joined_df['median_household_size'])  
)

In [22]:
# Perform Min-Max Normalization so that all are between 0 and 1
min_val = min(joined_df['demand_score'])
max_val = max(joined_df['demand_score'])

joined_df['demand_score'] = (joined_df['demand_score'] - min_val) / (max_val - min_val)

In [24]:
joined_df[joined_df['demand_score'] == joined_df['demand_score'].max()]

,zip,lat,lng,population,density,median_household_income,median_age,median_household_size,demand_score
2467,10025,40.79825,-73.96831,93223.0,40494.2,103440.0,40.0,2.21,1.0
